# 第13章 金融文本与自然语言处理 —— 配套代码

对应正文 `book/part3/13-nlp-text.md`。

## 演示内容

1. 环境初始化
2. 硬编码中文财经句子数据集（50条带标签）
3. jieba 分词与停用词过滤展示
4. 词频统计与高频词可视化
5. TF-IDF 向量化（unigram + bigram）
6. 逻辑回归 + 朴素贝叶斯情感分类（5折CV）
7. 混淆矩阵与逻辑回归系数解读
8. 情感词典法打分（对比）
9. 两种方法可视化对比
10. （可选）FinBERT 推理（联网保护）
11. 情感-收益时间对齐演示
12. 超参数敏感性分析
13. 本章小结
14. 习题参考解答

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1: 环境初始化
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import math

import jieba
import jieba.posseg as pseg

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from scipy.stats import pearsonr, spearmanr

from fds import set_chinese_font
set_chinese_font()
np.random.seed(42)

import sklearn
print('环境初始化完成')
print(f'jieba 版本: {jieba.__version__}')
print(f'scikit-learn 版本: {sklearn.__version__}')

## 13.1 硬编码中文财经句子数据集

本章在 notebook 内**硬编码**一个由 50 条中文财经句子组成的小型标注数据集：

- **标签 1（正面/利好）**：业绩超预期、股价创新高、大额回购、利润大增等
- **标签 0（负面/利空）**：亏损扩大、监管问询、债务风险、业绩暴雷等

> 此数据集足以演示完整的 NLP 情感分类流程。真实项目应收集 1000 条以上标注数据。

In [ ]:
# Cell 2: 硬编码中文财经句子数据集（50条）

# ---- 正面（利好）样本 25 条 ----
positive_samples = [
    ("公司净利润同比大增40%，远超市场预期", 1),
    ("季报显示营收创历史新高，增长强劲", 1),
    ("公司宣布大额回购计划，彰显管理层信心", 1),
    ("业绩预增公告发布，预计全年净利润翻倍", 1),
    ("公司成功签署百亿战略合作协议，前景广阔", 1),
    ("分红方案超预期，每股分红大幅提升", 1),
    ("公司获得重大专利授权，核心竞争力增强", 1),
    ("营业收入同比增长35%，盈利能力显著改善", 1),
    ("公司成功完成定增融资，资金实力雄厚", 1),
    ("股价突破历史新高，北向资金持续净买入", 1),
    ("公司主营业务快速增长，市占率持续提升", 1),
    ("扭亏为盈，归母净利润同比大幅增长", 1),
    ("获得行业最高评级，研报一致推荐买入", 1),
    ("核心产品销量爆发，供不应求", 1),
    ("公司新产品发布获市场热烈反响，订单激增", 1),
    ("战略合作落地，预期带来数十亿营收增量", 1),
    ("公司管理层增持，彰显对未来发展的信心", 1),
    ("毛利率显著提升，降本增效成果显著", 1),
    ("公司荣获国家高新技术企业认定，政策红利明显", 1),
    ("出口订单大幅增长，国际市场持续扩张", 1),
    ("公司主力产品获批，市场空间巨大", 1),
    ("现金流充裕，财务状况持续健康", 1),
    ("客户续约率超95%，业务黏性强劲", 1),
    ("公司IPO成功上市，融资规模超预期", 1),
    ("业绩持续高增长，估值已处历史低位", 1),
]

# ---- 负面（利空）样本 25 条 ----
negative_samples = [
    ("公司净利润同比下滑60%，业绩严重不及预期", 0),
    ("公司因财务造假被证监会立案调查", 0),
    ("季报显示亏损扩大，现金流持续告急", 0),
    ("公司收到交易所问询函，要求说明资金占用问题", 0),
    ("大股东减持套现，持股比例大幅下降", 0),
    ("债务危机爆发，公司面临流动性风险", 0),
    ("主营业务收入同比下降25%，盈利能力堪忧", 0),
    ("公司被列入ST名单，面临退市风险", 0),
    ("核心管理层集体离职，公司治理存在重大隐患", 0),
    ("产品质量事故频发，召回损失惨重", 0),
    ("公司涉及重大诉讼，或遭巨额赔偿", 0),
    ("营收大幅不及预期，多家机构下调评级", 0),
    ("公司违规担保事项曝光，风险敞口巨大", 0),
    ("市场份额持续下滑，竞争对手快速蚕食", 0),
    ("公司实控人被采取强制措施，经营面临不确定性", 0),
    ("利润预警：预计亏损超5亿元，触发退市警示", 0),
    ("银行抽贷断供，公司资金链面临断裂风险", 0),
    ("公司存货大幅积压，减值压力沉重", 0),
    ("行业政策收紧，公司核心业务受重大冲击", 0),
    ("应收账款坏账风险激增，财务质量堪忧", 0),
    ("公司被曝数据造假，市场信任度骤降", 0),
    ("多名独立董事相继辞职，公司内控存疑", 0),
    ("公司商誉减值损失超预期，净利润大幅缩水", 0),
    ("行业需求持续萎缩，公司业绩承压明显", 0),
    ("公司经营性现金流连续三季度为负，风险上升", 0),
]

all_samples = positive_samples + negative_samples
texts  = [s[0] for s in all_samples]
labels = [s[1] for s in all_samples]

df = pd.DataFrame({'text': texts, 'label': labels})
df['label_name'] = df['label'].map({1: '正面(利好)', 0: '负面(利空)'})

print(f'数据集大小: {len(df)} 条')
print(f'正面样本: {df["label"].sum()} 条')
print(f'负面样本: {(df["label"]==0).sum()} 条')
print()
print('示例正面样本:')
for t in df[df['label']==1]['text'].head(3):
    print(f'  [正] {t}')
print('示例负面样本:')
for t in df[df['label']==0]['text'].head(3):
    print(f'  [负] {t}')
df[['text', 'label', 'label_name']].head()

## 13.2 jieba 分词与停用词过滤

中文文本处理的核心：jieba 使用基于 HMM 的分词算法，对中文财经文本效果良好。

**停用词策略**：
- 语言停用词：的、了、在、是、和……
- 金融通用词：公司、市场、业务（对情感判断无帮助）
- 单字词：通常噪声大，去除

In [ ]:
# Cell 3: jieba 分词展示 + 停用词过滤

# 金融文本停用词表（精简版）
STOPWORDS = {
    "的", "了", "在", "是", "和", "与", "对", "将", "为",
    "也", "不", "而", "但", "从", "到", "由", "于", "等",
    "该", "其", "以", "因此", "然而", "可以", "已", "后",
    "前", "中", "上", "下", "有", "被", "使", "让",
    # 金融通用词（对区分情感无帮助）
    "公司", "市场", "业务", "发展", "进行", "开展", "相关",
    "方面", "情况", "工作", "表示", "同时", "目前", "近期",
}

def tokenize(text, stopwords=STOPWORDS):
    '''jieba分词 -> 去停用词 -> 去单字'''
    text_clean = re.sub(r'[^一-鿿]', ' ', text)
    words = jieba.lcut(text_clean)
    return [w for w in words if w not in stopwords and len(w) > 1]

# 演示分词效果
demo_texts = [
    "公司净利润同比大增40%，远超市场预期",
    "季报显示亏损扩大，现金流持续告急",
    "大股东减持套现，持股比例大幅下降",
]

print("=== jieba 分词演示 ===\n")
for text in demo_texts:
    raw_words = jieba.lcut(text)
    filtered = tokenize(text)
    print(f"原文: {text}")
    print(f"  全词: {raw_words}")
    print(f"  去停+去单字: {filtered}")
    print()

# 对全数据集分词
df['tokens'] = df['text'].apply(tokenize)
df['token_str'] = df['tokens'].apply(lambda x: ' '.join(x))

print(f"分词完成，平均每句 {df['tokens'].apply(len).mean():.1f} 个词")
print("\n前5行分词结果:")
print(df[['text', 'tokens', 'label_name']].head().to_string(index=False))

# 词性标注演示
print("\n=== 词性标注演示 (jieba.posseg) ===")
demo_pos = "业绩持续高增长，估值已处历史低位"
tag_map = {'n': '名词', 'v': '动词', 'a': '形容词', 'd': '副词', 'm': '数词'}
for word, flag in pseg.lcut(demo_pos):
    if len(word) > 1:
        tag_cn = tag_map.get(flag[0], flag)
        print(f"  {word:6s}  [{flag}] {tag_cn}")


## 13.3 词频统计与高频词可视化

在建模前，先了解正面和负面样本中最常见的词汇——也是情感词典构建的基础。

In [ ]:
# Cell 4: 词频统计 + 高频词条形图

pos_words = [w for tokens in df[df['label']==1]['tokens'] for w in tokens]
neg_words = [w for tokens in df[df['label']==0]['tokens'] for w in tokens]

pos_counter = Counter(pos_words)
neg_counter = Counter(neg_words)

TOP_N = 15
top_pos = pos_counter.most_common(TOP_N)
top_neg = neg_counter.most_common(TOP_N)

print(f"正面样本高频词 Top{TOP_N}:")
for w, c in top_pos:
    print(f"  {w:8s} {c:3d} 次")
print()
print(f"负面样本高频词 Top{TOP_N}:")
for w, c in top_neg:
    print(f"  {w:8s} {c:3d} 次")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

words_p, counts_p = zip(*top_pos)
axes[0].barh(range(TOP_N), list(counts_p[::-1]),
             color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_yticks(range(TOP_N))
axes[0].set_yticklabels(list(words_p[::-1]), fontsize=11)
axes[0].set_xlabel('出现次数')
axes[0].set_title(f'正面(利好)样本 Top{TOP_N} 高频词', fontsize=13, color='steelblue')
axes[0].grid(axis='x', alpha=0.3)

words_n, counts_n = zip(*top_neg)
axes[1].barh(range(TOP_N), list(counts_n[::-1]),
             color='crimson', alpha=0.8, edgecolor='white')
axes[1].set_yticks(range(TOP_N))
axes[1].set_yticklabels(list(words_n[::-1]), fontsize=11)
axes[1].set_xlabel('出现次数')
axes[1].set_title(f'负面(利空)样本 Top{TOP_N} 高频词', fontsize=13, color='crimson')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('正面 vs 负面样本高频词对比', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 13.4 TF-IDF 向量化

$$\text{TF-IDF}(w, d) = \frac{c_{w,d}}{\sum_{w'} c_{w',d}} \times \log \frac{N}{1 + df_w}$$

- $N$：文档总数；$df_w$：包含词 $w$ 的文档数
- `ngram_range=(1,2)`：同时提取 unigram 和 bigram，捕获"大幅增长"等短语
- `sublinear_tf=True`：对词频取 $\log(1+tf)$ 平滑，减少高频词主导

In [ ]:
# Cell 5: TF-IDF 向量化

vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x.split(),
    token_pattern=None,
    ngram_range=(1, 2),      # unigram + bigram
    max_features=500,
    min_df=1,
    sublinear_tf=True,       # log(1+tf)
)

X = vectorizer.fit_transform(df['token_str'])
y = np.array(df['label'])
feature_names = vectorizer.get_feature_names_out()

print(f"TF-IDF 矩阵形状: {X.shape}")
print(f"  样本数: {X.shape[0]}  特征数(词/短语): {X.shape[1]}")
print(f"  矩阵稀疏度: {1 - X.nnz / (X.shape[0]*X.shape[1]):.1%}")

# 按类别展示 TF-IDF 权重最高的词
X_dense = X.toarray()
pos_mask = (y == 1)
neg_mask = (y == 0)

top_pos_tfidf = pd.Series(
    X_dense[pos_mask].mean(axis=0), index=feature_names
).sort_values(ascending=False).head(10)

top_neg_tfidf = pd.Series(
    X_dense[neg_mask].mean(axis=0), index=feature_names
).sort_values(ascending=False).head(10)

print("\n正面样本 TF-IDF 权重最高的词/短语:")
print(top_pos_tfidf.round(4).to_string())
print("\n负面样本 TF-IDF 权重最高的词/短语:")
print(top_neg_tfidf.round(4).to_string())


## 13.5 情感分类：逻辑回归 + 朴素贝叶斯

**逻辑回归**：
$$P(y=1|\mathbf{x}) = \frac{1}{1+e^{-(\boldsymbol{\beta}^T\mathbf{x}+\beta_0)}}$$

**朴素贝叶斯**：假设各词在给定类别下条件独立，$$P(y=1|\mathbf{x}) \propto P(y=1)\prod_j P(x_j|y=1)$$

使用 **StratifiedKFold** 保持正负样本比例均衡。

In [ ]:
# Cell 6: 逻辑回归 + 朴素贝叶斯（训练集/测试集 + 5折CV）

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"训练集: {X_train.shape[0]} 条 | 测试集: {X_test.shape[0]} 条")
print(f"训练集正面占比: {y_train.mean():.1%} | 测试集正面占比: {y_test.mean():.1%}")

# ---- 模型 A: 逻辑回归 ----
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs')
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

# ---- 模型 B: 多项式朴素贝叶斯 ----
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)

# ---- 5折交叉验证 ----
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_lr = cross_val_score(
    LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs'),
    X, y, cv=skf, scoring='accuracy'
)
cv_nb = cross_val_score(MultinomialNB(alpha=1.0), X, y, cv=skf, scoring='accuracy')

print()
print("=" * 52)
print("           分类器性能对比")
print("=" * 52)
print(f"{'方法':<20} {'测试集准确率':>12} {'5折CV均值':>10} {'CV标准差':>10}")
print("-" * 52)
print(f"{'逻辑回归':<20} {acc_lr:>12.4f} {cv_lr.mean():>10.4f} {cv_lr.std():>10.4f}")
print(f"{'朴素贝叶斯':<20} {acc_nb:>12.4f} {cv_nb.mean():>10.4f} {cv_nb.std():>10.4f}")
print()
print("=== 逻辑回归详细分类报告 ===")
print(classification_report(y_test, y_pred_lr, target_names=['负面(利空)', '正面(利好)']))
print("=== 朴素贝叶斯详细分类报告 ===")
print(classification_report(y_test, y_pred_nb, target_names=['负面(利空)', '正面(利好)']))


## 13.6 混淆矩阵与逻辑回归系数

混淆矩阵直观展示分类器的错误类型：

| | 预测负面 | 预测正面 |
|---|---|---|
| **真实负面** | TN（正确） | FP（误报） |
| **真实正面** | FN（漏报） | TP（正确） |

逻辑回归系数绝对值最大的词是情感判断的**关键特征**，可用于验证模型可解释性。

In [ ]:
# Cell 7: 混淆矩阵可视化 + 逻辑回归系数解读

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
class_names = ['负面(利空)', '正面(利好)']

for ax, y_pred, title, cmap in zip(
    axes,
    [y_pred_lr, y_pred_nb],
    [f'逻辑回归 (Acc={acc_lr:.2%})', f'朴素贝叶斯 (Acc={acc_nb:.2%})'],
    ['Blues', 'Oranges']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('预测标签', fontsize=11)
    ax.set_ylabel('真实标签', fontsize=11)

plt.suptitle('情感分类混淆矩阵', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 逻辑回归系数：最强正负面信号词
coef = lr_model.coef_[0]
top_pos_idx = np.argsort(coef)[-15:][::-1]
top_neg_idx = np.argsort(coef)[:15]

print("=== 逻辑回归：最强正面信号词（系数最大）===")
for idx in top_pos_idx:
    print(f"  {feature_names[idx]:12s}  系数={coef[idx]:.4f}")
print()
print("=== 逻辑回归：最强负面信号词（系数最小）===")
for idx in top_neg_idx:
    print(f"  {feature_names[idx]:12s}  系数={coef[idx]:.4f}")

# 可视化系数
fig2, ax2 = plt.subplots(figsize=(12, 6))
top_pos_words = [feature_names[i] for i in top_pos_idx[:10]]
top_pos_coefs = [coef[i] for i in top_pos_idx[:10]]
top_neg_words = [feature_names[i] for i in top_neg_idx[:10]]
top_neg_coefs = [coef[i] for i in top_neg_idx[:10]]

all_words = top_neg_words[::-1] + top_pos_words
all_coefs = top_neg_coefs[::-1] + top_pos_coefs
colors = ['crimson']*10 + ['steelblue']*10

ax2.barh(range(20), all_coefs, color=colors, alpha=0.8, edgecolor='white')
ax2.set_yticks(range(20))
ax2.set_yticklabels(all_words, fontsize=10)
ax2.axvline(0, color='black', lw=1)
ax2.set_xlabel('逻辑回归系数', fontsize=12)
ax2.set_title('情感分类：关键词系数（蓝=正面信号，红=负面信号）', fontsize=13)
ax2.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 13.7 情感词典法打分

词典法**无需训练数据**，直接用正/负面词表计分：

$$\text{Score} = \frac{N_{\text{pos}} - N_{\text{neg}}}{N_{\text{pos}} + N_{\text{neg}} + 1} \in (-1, 1)$$

- 正值 → 正面情感
- 负值 → 负面情感
- 0 附近 → 中性

**主要局限**：无法处理否定句（"没有增长"）、无法感知语境。

In [ ]:
# Cell 8: 情感词典法打分

# 金融领域简化情感词典
POSITIVE_WORDS = {
    "增长", "上涨", "利好", "超预期", "大增", "盈利", "突破",
    "新高", "亮眼", "扭亏", "稳健", "回升", "强劲", "增持",
    "回购", "分红", "翻倍", "获批", "激增", "提升", "改善",
    "雄厚", "扩张", "充裕", "健康", "领涨", "成功"
}

NEGATIVE_WORDS = {
    "下跌", "亏损", "暴雷", "问询", "风险", "违规", "诉讼",
    "下滑", "减持", "警示", "退市", "处罚", "欺诈", "造假",
    "下降", "债务", "危机", "断裂", "积压", "减值", "萎缩",
    "承压", "告急", "冲击", "离职", "调查", "立案", "收紧"
}

def dict_score(text):
    '''词典法情感打分: (-1, 1)'''
    words = tokenize(text)
    n_pos = sum(1 for w in words if w in POSITIVE_WORDS)
    n_neg = sum(1 for w in words if w in NEGATIVE_WORDS)
    return (n_pos - n_neg) / (n_pos + n_neg + 1)

def dict_predict(text, threshold=0.0):
    '''词典法预测: score > threshold -> 1 (正面), 否则 0 (负面)'''
    return int(dict_score(text) > threshold)

df['dict_score'] = df['text'].apply(dict_score)
df['dict_pred']  = df['text'].apply(dict_predict)
acc_dict = accuracy_score(df['label'], df['dict_pred'])

print(f"情感词典法准确率（全集）: {acc_dict:.4f}")
print()
print("词典法分类报告:")
print(classification_report(df['label'], df['dict_pred'], target_names=['负面(利空)', '正面(利好)']))

# 打印典型样本得分
print("=== 典型样本词典得分 ===")
for _, row in df.sample(10, random_state=42).sort_values('label').iterrows():
    label_str = '[正面]' if row['label'] == 1 else '[负面]'
    correct = 'OK' if row['dict_pred'] == row['label'] else 'ERR'
    print(f"{correct} {label_str} {row['dict_score']:+.3f}  {row['text'][:35]}")


## 13.8 两种方法可视化对比

将词典法情感得分和逻辑回归预测概率并排可视化，对比两种方法在正/负面样本上的区分度。

In [ ]:
# Cell 9: 方法可视化对比

X_all_vec = vectorizer.transform(df['token_str'])
prob_lr_all = lr_model.predict_proba(X_all_vec)[:, 1]
pred_lr_all = lr_model.predict(X_all_vec)
acc_lr_all = accuracy_score(df['label'], pred_lr_all)
df['lr_prob'] = prob_lr_all

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 词典法得分分布
pos_scores = df[df['label']==1]['dict_score']
neg_scores = df[df['label']==0]['dict_score']
axes[0].hist(neg_scores.values, bins=12, alpha=0.7, color='crimson',
             label='负面(利空)', density=True)
axes[0].hist(pos_scores.values, bins=12, alpha=0.7, color='steelblue',
             label='正面(利好)', density=True)
axes[0].axvline(0, color='black', lw=2, linestyle='--', label='判决边界(0)')
axes[0].set_xlabel('词典情感得分', fontsize=12)
axes[0].set_ylabel('概率密度')
axes[0].set_title(f'词典法情感得分分布\n(准确率={acc_dict:.2%})', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# 逻辑回归预测概率分布
pos_probs = df[df['label']==1]['lr_prob']
neg_probs = df[df['label']==0]['lr_prob']
axes[1].hist(neg_probs.values, bins=12, alpha=0.7, color='crimson',
             label='负面(利空)', density=True)
axes[1].hist(pos_probs.values, bins=12, alpha=0.7, color='steelblue',
             label='正面(利好)', density=True)
axes[1].axvline(0.5, color='black', lw=2, linestyle='--', label='判决边界(0.5)')
axes[1].set_xlabel('逻辑回归正面概率 P(正面)', fontsize=12)
axes[1].set_ylabel('概率密度')
axes[1].set_title(f'逻辑回归预测概率分布\n(准确率={acc_lr_all:.2%}，全集参考)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('情感分析方法对比：词典法 vs 逻辑回归', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("=== 三种方法准确率汇总 ===")
print(f"{'方法':<25} {'准确率':>10}")
print("-" * 38)
print(f"{'情感词典法':<25} {acc_dict:>10.4f}")
print(f"{'逻辑回归(测试集)':<25} {acc_lr:>10.4f}")
print(f"{'逻辑回归(全集,参考)':<25} {acc_lr_all:>10.4f}")
print(f"{'朴素贝叶斯(测试集)':<25} {acc_nb:>10.4f}")


## 13.9 （可选）预训练模型：中文 FinBERT

> **此 cell 需联网下载约 400MB 预训练模型**。程序用 `try/except` 包裹，若失败自动跳过，不影响其他 cell。

BERT 自注意力公式：
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

中文 FinBERT 在金融语料上继续预训练，专业术语和上下文感知能力显著优于 TF-IDF 方法。

In [ ]:
# Cell 10: （可选）中文 FinBERT 推理 —— try/except 保护

FINBERT_AVAILABLE = False
classifier = None

try:
    from transformers import pipeline as hf_pipeline
    print("尝试从本地缓存加载 FinBERT 模型...")
    model_name = "IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment"
    classifier = hf_pipeline(
        "text-classification",
        model=model_name,
        local_files_only=True,
    )
    FINBERT_AVAILABLE = True
    print("FinBERT 模型加载成功（本地缓存）")

except Exception as e:
    print(f"FinBERT 不可用 ({type(e).__name__})")
    print("  原因：模型未下载或无网络，已自动跳过。")
    print("  如需使用，请在联网环境下运行：")
    print("    from transformers import pipeline")
    print("    pipeline('text-classification',")
    print("             model='IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment')")

if FINBERT_AVAILABLE and classifier is not None:
    test_sents = [
        "公司净利润同比大增40%，远超市场预期",
        "公司因财务造假被证监会立案调查",
        "营业收入同比增长35%，盈利能力显著改善",
        "公司被列入ST名单，面临退市风险",
    ]
    print("\nFinBERT 推理结果：")
    for sent in test_sents:
        result = classifier(sent[:128])[0]
        print(f"  {result['label']:10s} ({result['score']:.4f})  {sent}")
else:
    print("\n[跳过] FinBERT 推理演示")
    print("\n理论要点：")
    print("  - BERT 双向 Transformer，能感知否定与上下文")
    print("  - FinBERT 在金融语料预训练，专业术语更准确")
    print("  - 典型准确率: 词典法~65% | 逻辑回归~80% | FinBERT微调~90%+")


## 13.10 从情感分数到投资信号：时间对齐

$$\text{IC}_t = \text{Corr}(\hat{S}_{i,t-1},\, r_{i,t})$$

- $\hat{S}_{i,t-1}$：$t-1$ 日的情感因子（`shift(1)`）
- $r_{i,t}$：$t$ 日实际收益率

**黄金法则**：情感分数必须 `shift(1)` 才能作为预测因子，否则构成前视偏差。

In [ ]:
# Cell 11: 情感-收益时间对齐演示（模拟数据）

np.random.seed(42)
N_DAYS = 200

# 模拟情感分数序列
sentiment_true = np.random.normal(0, 1, N_DAYS)
noise = np.random.normal(0, 2, N_DAYS)
returns_sim = 0.02 * sentiment_true + noise
returns_sim /= returns_sim.std()

dates = pd.date_range('2022-01-01', periods=N_DAYS, freq='B')
df_sim = pd.DataFrame({
    'sentiment': sentiment_true,
    'return_t': returns_sim,
}, index=dates)

df_sim['sentiment_lag1'] = df_sim['sentiment'].shift(1)
df_sim['return_next']    = df_sim['return_t'].shift(-1)
df_clean = df_sim.dropna()

# IC（正确做法：lag-1 因子）
ic_val,  _ = pearsonr(df_clean['sentiment_lag1'], df_clean['return_next'])
ric_val, _ = spearmanr(df_clean['sentiment_lag1'], df_clean['return_next'])

# IC（错误做法：同期 → 前视偏差）
ic_biased, _ = pearsonr(df_clean['sentiment'], df_clean['return_t'])

print("=== 情感-收益相关性分析 ===")
print(f"正确做法 (lag-1) IC     : {ic_val:+.4f}")
print(f"正确做法 (lag-1) RankIC : {ric_val:+.4f}")
print(f"前视偏差 (同期)  IC     : {ic_biased:+.4f}  <- 错误！虚高")
print(f"虚高幅度: {ic_biased - ic_val:+.4f}")

# 散点图对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, x_col, y_col, ic, title, color in [
    (axes[0], 'sentiment_lag1', 'return_next', ic_val,
     f'正确: 情感(t) vs 收益(t+1)\nIC={ic_val:.4f}（无前视偏差）', 'steelblue'),
    (axes[1], 'sentiment', 'return_t', ic_biased,
     f'错误: 情感(t) vs 收益(t)  \nIC={ic_biased:.4f}（前视偏差！）', 'crimson'),
]:
    ax.scatter(df_clean[x_col], df_clean[y_col], alpha=0.4, s=20, color=color)
    z = np.polyfit(df_clean[x_col], df_clean[y_col], 1)
    xr = np.linspace(df_clean[x_col].min(), df_clean[x_col].max(), 100)
    ax.plot(xr, np.poly1d(z)(xr), 'k-', lw=2)
    ax.set_xlabel('情感分数', fontsize=11)
    ax.set_ylabel('收益率', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.3)

plt.suptitle('情感信号时间对齐：正确(shift-1) vs 前视偏差', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 13.11 超参数敏感性分析

系统探索 TF-IDF + 逻辑回归的关键超参数对准确率的影响：

| 超参数 | 含义 | 典型范围 |
|--------|------|--------|
| `ngram_range` | N-gram 范围 | (1,1), (1,2), (1,3) |
| `max_features` | 词汇表大小 | 100 ~ 5000 |
| `C`（LR） | 正则化倒数 | 0.01 ~ 10 |

In [ ]:
# Cell 12: 超参数敏感性分析（N-gram + 正则化 C）

skf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_grid = []

for ng in [(1,1), (1,2), (1,3)]:
    for mf in [100, 200, 500]:
        vec_tmp = TfidfVectorizer(
            tokenizer=lambda x: x.split(), token_pattern=None,
            ngram_range=ng, max_features=mf,
            sublinear_tf=True, min_df=1
        )
        X_tmp = vec_tmp.fit_transform(df['token_str'])
        lr_tmp = LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs')
        nb_tmp = MultinomialNB(alpha=1.0)
        cv_lr = cross_val_score(lr_tmp, X_tmp, y, cv=skf_inner, scoring='accuracy').mean()
        cv_nb = cross_val_score(nb_tmp, X_tmp, y, cv=skf_inner, scoring='accuracy').mean()
        results_grid.append({
            'ngram': str(ng), 'max_features': mf,
            'LR 5折CV': round(cv_lr, 4),
            'NB 5折CV': round(cv_nb, 4),
        })

df_grid = pd.DataFrame(results_grid)
print("=== 超参数网格搜索结果（5折CV准确率）===")
print(df_grid.to_string(index=False))

# 正则化 C 的影响
C_vals = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
cv_by_C = []
for c_val in C_vals:
    lr_c = LogisticRegression(C=c_val, max_iter=1000, random_state=42, solver='lbfgs')
    cv_by_C.append(cross_val_score(lr_c, X, y, cv=skf_inner, scoring='accuracy').mean())

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_lr = df_grid.pivot(index='max_features', columns='ngram', values='LR 5折CV')
pivot_lr.plot(kind='bar', ax=axes[0], alpha=0.8, edgecolor='white')
axes[0].set_xlabel('max_features')
axes[0].set_ylabel('5折CV准确率')
axes[0].set_title('N-gram 与词汇量 vs 准确率（逻辑回归）', fontsize=12)
axes[0].legend(title='ngram_range', fontsize=9)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(np.log10(C_vals), cv_by_C, 'o-', color='steelblue', lw=2, markersize=8)
for c_val, cv_v in zip(C_vals, cv_by_C):
    axes[1].annotate(f'C={c_val}', (np.log10(c_val), cv_v + 0.005), fontsize=8, ha='center')
axes[1].set_xlabel('log10(C)  (C小→强正则化)', fontsize=11)
axes[1].set_ylabel('5折CV准确率')
axes[1].set_title('逻辑回归正则化强度 vs 准确率', fontsize=12)
axes[1].grid(alpha=0.3)

plt.suptitle('TF-IDF + 逻辑回归超参数敏感性分析', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 13.11b TF-IDF 特征权重双类别对比

将正面/负面样本中 TF-IDF 权重最高的特征可视化，直观展示两类情感的判别特征。这与逻辑回归系数分析形成互补：
TF-IDF 权重反映词汇**本身的区分度**，系数反映模型**学到的权重**。

In [ ]:
# Cell 13: TF-IDF 特征重要性双类别可视化

# 正负面样本平均TF-IDF权重 (已在Cell 5计算过X_dense)
feat_pos = pd.Series(X_dense[pos_mask].mean(axis=0), index=feature_names)
feat_neg = pd.Series(X_dense[neg_mask].mean(axis=0), index=feature_names)

# 差值：正面权重 - 负面权重，绝对值越大越有区分力
feat_diff = feat_pos - feat_neg
top_pos_diff = feat_diff.sort_values(ascending=False).head(12)
top_neg_diff = feat_diff.sort_values(ascending=True).head(12)

print("=== TF-IDF 差值最大的特征（正面区分词）===")
print(top_pos_diff.round(4).to_string())
print("\n=== TF-IDF 差值最小的特征（负面区分词）===")
print(top_neg_diff.round(4).to_string())

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 正面区分词（蓝色条）
words_pd = top_pos_diff.index[::-1]
vals_pd  = top_pos_diff.values[::-1]
axes[0].barh(range(len(words_pd)), vals_pd, color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(words_pd)))
axes[0].set_yticklabels(list(words_pd), fontsize=10)
axes[0].set_xlabel('TF-IDF 权重差（正面 - 负面）')
axes[0].set_title('正面区分特征\n（正面样本权重显著高于负面）', fontsize=12, color='steelblue')
axes[0].grid(axis='x', alpha=0.3)
axes[0].axvline(0, color='black', lw=0.8)

# 负面区分词（红色条）
words_nd = top_neg_diff.index
vals_nd  = top_neg_diff.values
axes[1].barh(range(len(words_nd)), vals_nd, color='crimson', alpha=0.8)
axes[1].set_yticks(range(len(words_nd)))
axes[1].set_yticklabels(list(words_nd), fontsize=10)
axes[1].set_xlabel('TF-IDF 权重差（正面 - 负面）')
axes[1].set_title('负面区分特征\n（负面样本权重显著高于正面）', fontsize=12, color='crimson')
axes[1].grid(axis='x', alpha=0.3)
axes[1].axvline(0, color='black', lw=0.8)

plt.suptitle('TF-IDF 特征区分度：正面 vs 负面情感词汇', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n=== 综合情感词汇对比 ===")
comparison = pd.DataFrame({
    '正面TF-IDF': feat_pos,
    '负面TF-IDF': feat_neg,
    '差值': feat_diff
}).sort_values('差值', ascending=False)
print("Top 10 正面词汇（差值最大）:")
print(comparison.head(10).round(4).to_string())
print("\nTop 10 负面词汇（差值最小）:")
print(comparison.tail(10).round(4).to_string())


## 13.11c 对新文本的情感预测

训练好的模型可以直接对新的财经句子进行情感预测，演示完整的推理流程。

In [ ]:
# Cell 14b: 对新财经文本的情感预测（完整推理流程）

# 准备新文本样本（未见过的句子）
new_texts = [
    "公司一季度收入同比增长28%，利润率创近三年新高",    # 预期：正面
    "主力机构持续减持，股价短期承压",                    # 预期：负面
    "公司与头部互联网平台签署战略合作协议",               # 预期：正面
    "公司被曝财务数据虚假，监管部门已启动调查程序",       # 预期：负面
    "受原材料成本上涨影响，公司毛利率有所下降",           # 预期：偏负面
    "公司成功完成债券置换，债务压力显著缓解",             # 预期：正面
]

# 完整推理流程
print("=== 新文本情感预测（逻辑回归 + 词典法）===\n")
print(f"{'LR预测':8} {'LR概率':8} {'词典法':8} {'文本'}")
print("-" * 75)

for text in new_texts:
    # 1. 分词
    tokens = tokenize(text)
    token_str = ' '.join(tokens)

    # 2. 逻辑回归预测
    X_new = vectorizer.transform([token_str])
    lr_label = lr_model.predict(X_new)[0]
    lr_prob  = lr_model.predict_proba(X_new)[0, 1]
    lr_str   = '正面(利好)' if lr_label == 1 else '负面(利空)'

    # 3. 词典法打分
    dscore = dict_score(text)
    dict_str = '正面(利好)' if dscore > 0 else '负面(利空)'

    agree = 'OK' if lr_label == (1 if dscore > 0 else 0) else '?? '
    print(f"[{lr_str}] P={lr_prob:.3f}  [{dict_str}] d={dscore:+.3f}  {agree}  {text[:35]}")

print("\n=== 关键词分析 ===")
for text in new_texts[:3]:
    tokens = tokenize(text)
    pos_hits = [w for w in tokens if w in POSITIVE_WORDS]
    neg_hits = [w for w in tokens if w in NEGATIVE_WORDS]
    print(f"\n文本: {text[:40]}")
    print(f"  正面词: {pos_hits}")
    print(f"  负面词: {neg_hits}")
    print(f"  分词: {tokens}")


## 13.12 本章小结

| 环节 | 关键方法 | 注意事项 |
|------|---------|--------|
| 分词 | jieba 精确模式 | 自定义金融词典 |
| 去噪 | 停用词 + 单字过滤 | 保留情感关键词 |
| 特征 | TF-IDF + bigram | max_features 避免过拟合 |
| 建模 | LR / NB 并用 | 交叉验证选优 |
| 评估 | Accuracy + F1 | 样本不均衡用 F1 |
| 信号 | 情感分数 shift(1) | 严禁前视偏差 |

> **核心原则**：文本情感信号必须严格先于预测收益，`shift(1)` 是金融文本因子的黄金法则。

## 习题参考解答

以下代码对应正文第 13.10 节的 5 道习题，可直接运行验证。

In [ ]:
# Cell 14: 习题 13.1 - 13.3 参考解答

print("=" * 60)
print("习题 13.1: 分词与停用词")
print("=" * 60)

text_q1 = "受宏观经济下行影响，公司主营业务收入同比下降15%，净利润亏损2亿元。"
raw_q1 = jieba.lcut(text_q1)
filtered_q1 = tokenize(text_q1)

print(f"\n原文: {text_q1}")
print(f"\n(a) jieba 分词结果:\n    {raw_q1}")
print(f"\n(b) 去停用词+去单字后:\n    {filtered_q1}")

key_neg_q1 = [w for w in filtered_q1 if w in NEGATIVE_WORDS]
print(f"\n(c) 情感判断: 负面")
print(f"    词典中的负面关键词: {key_neg_q1}")
print(f"    其他: 下行(负面语义), 下降(减少), 亏损(直接损失)")

print("\n" + "=" * 60)
print("习题 13.2: TF-IDF 直觉")
print("=" * 60)

docs_q2 = [
    "净利润 大幅 增长 超出 预期",
    "净利润 同比 下滑 低于 预期",
    "新品 市场 反应 良好",
]
N_q2 = len(docs_q2)
df_word_q2 = sum(1 for d in docs_q2 if "预期" in d.split())
idf_yuqi = math.log(N_q2 / (1 + df_word_q2))
print(f"\n(a) IDF('预期') = log({N_q2}/{1+df_word_q2}) = {idf_yuqi:.4f}")
print(f"    '预期'出现在{df_word_q2}个文档中，IDF接近0，权重低")
print(f"\n(b) '公司'几乎出现在所有财报中，df≈N，IDF→0，TF-IDF≈0")
print(f"    应加入停用词表")
print(f"\n(c) 建议加入停用词: 公司, 市场, 发展, 情况, 进行, 业务")

print("\n" + "=" * 60)
print("习题 13.3: 词典法 vs 有监督分类")
print("=" * 60)

print(f"\n(a) 准确率对比:")
print(f"    情感词典法      : {acc_dict:.4f}")
print(f"    逻辑回归(测试集): {acc_lr:.4f}")
print(f"    朴素贝叶斯(测试集): {acc_nb:.4f}")

print(f"\n(b) 词典法典型错误（误分类样本）:")
errors_dict = df[df['dict_pred'] != df['label']][['text', 'label', 'dict_pred', 'dict_score']]
for _, row in errors_dict.head(3).iterrows():
    true_str = '正面' if row['label'] == 1 else '负面'
    pred_str = '正面' if row['dict_pred'] == 1 else '负面'
    print(f"  真实:{true_str} 预测:{pred_str} 得分:{row['dict_score']:+.3f}  {row['text'][:30]}")

print(f"\n(c) 提升词典法精度的方法:")
print(f"  1. 否定处理: 检测否定词('不/没/未')前缀，翻转情感极性")
print(f"  2. 程度副词: '大幅/严重'=2x权重, '略微/稍许'=0.5x权重")
print(f"  3. 扩充词典: 从错误样本中人工提取新的情感词")
print(f"  4. 领域词典: 引入 NTUSD-Fin 或 Loughran-McDonald 中文版")


In [ ]:
# Cell 15: 习题 13.4 - 13.5 参考解答

print("=" * 60)
print("习题 13.4: 前视偏差识别")
print("=" * 60)

cases = [
    ("(a)", "2023-12-31收盘后年报 -> 2024-01-02开盘收益",
     "无前视偏差", "收盘后信息可用于次日交易，时间先后正确"),
    ("(b)", "2023-03-15 14:00研报 -> 当日收盘收益(15:00)",
     "有前视偏差！", "14:00研报影响当日15:00收盘价，存在时间重叠"),
    ("(c)", "shift(1)情感均值 -> 次日截面IC分析",
     "无前视偏差", "shift(1)确保情感因子在收益发生前已知"),
]
for cid, desc, verdict, reason in cases:
    print(f"\n{cid} {desc}")
    print(f"    判断: {verdict}")
    print(f"    理由: {reason}")

print("\n" + "=" * 60)
print("习题 13.5: 实战参数调整")
print("=" * 60)

skf_q5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n(b) ngram_range 对准确率的影响（5折CV）:")
for ng_q5 in [(1,1), (1,2), (1,3)]:
    vec_q5 = TfidfVectorizer(
        tokenizer=lambda x: x.split(), token_pattern=None,
        ngram_range=ng_q5, max_features=500,
        sublinear_tf=True, min_df=1
    )
    X_q5 = vec_q5.fit_transform(df['token_str'])
    lr_q5 = LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs')
    cv_q5 = cross_val_score(lr_q5, X_q5, y, cv=skf_q5, scoring='accuracy')
    print(f"  ngram={ng_q5}: CV={cv_q5.mean():.4f}+-{cv_q5.std():.4f}  (特征数={X_q5.shape[1]})")

print("\n  解释: bigram(1,2)通常最优；trigram在小数据集易过拟合")

print("\n(c) class_weight='balanced' 对混淆矩阵的影响:")
lr_bal = LogisticRegression(
    C=1.0, max_iter=1000, random_state=42,
    class_weight='balanced', solver='lbfgs'
)
lr_bal.fit(X_train, y_train)
y_pred_bal = lr_bal.predict(X_test)

cm_orig = confusion_matrix(y_test, y_pred_lr)
cm_bal  = confusion_matrix(y_test, y_pred_bal)
acc_bal = accuracy_score(y_test, y_pred_bal)

print(f"\n  原始 LR 混淆矩阵:\n  {cm_orig}")
print(f"\n  balanced LR 混淆矩阵:\n  {cm_bal}")
print(f"\n  准确率: 原始={acc_lr:.4f}  balanced={acc_bal:.4f}")
print("  balanced 提升少数类召回率，适合正负样本不均衡场景")


## 全章小结

本 notebook 完整演示了**中文金融文本情感分析**的端到端流程：

1. **数据**：50 条硬编码中文财经句子（正面/负面各 25 条），全离线
2. **预处理**：jieba 分词 + 停用词过滤 + 词性分析
3. **词频可视化**：高频词条形图揭示正负面语料差异
4. **TF-IDF 建模**：`ngram_range=(1,2)` 捕获短语语义，`sublinear_tf` 平滑
5. **分类器**：逻辑回归 + 朴素贝叶斯，5折 StratifiedKFold 交叉验证
6. **词典法**：简化金融情感词典打分，与监督方法对比
7. **时间对齐**：模拟数据演示 `shift(1)` 的重要性
8. **超参数分析**：N-gram 范围和正则化强度的影响

**关键结论**：
- 小数据集（50 条）上：逻辑回归 ≥ 朴素贝叶斯 > 词典法
- bigram 通常优于 unigram；trigram 在小数据集易过拟合
- `shift(1)` 是文本因子构建的核心约束
- 真实项目建议收集 1000+ 条标注样本，配合 FinBERT 微调可达 90%+ 准确率